In [1]:
import zipfile
import os

zip_path = "/content/sms+spam+collection.zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("/content/sms_dataset")

print("Dataset extracted successfully!")

Dataset extracted successfully!


In [2]:
import os

print(os.listdir("/content/sms_dataset"))

['SMSSpamCollection', 'readme']


In [3]:
import pandas as pd

file_path = "/content/sms_dataset/SMSSpamCollection"

df = pd.read_csv(
    file_path,
    sep="\t",
    header=None,
    names=["label", "message"]
)

print(df.head())

  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...


In [4]:
print("Shape of dataset:", df.shape)

print("\nColumn names:")
print(df.columns)

print("\nNumber of each label:")
print(df["label"].value_counts())

Shape of dataset: (5572, 2)

Column names:
Index(['label', 'message'], dtype='object')

Number of each label:
label
ham     4825
spam     747
Name: count, dtype: int64


In [5]:
# Check for missing values
print("Missing values:")
print(df.isnull().sum())

# Remove duplicate messages
df = df.drop_duplicates()

# Convert messages to lowercase
df["message"] = df["message"].str.lower()

print("\nDataset after preprocessing:")
print(df.head())

print("\nNew dataset shape:", df.shape)

Missing values:
label      0
message    0
dtype: int64

Dataset after preprocessing:
  label                                            message
0   ham  go until jurong point, crazy.. available only ...
1   ham                      ok lar... joking wif u oni...
2  spam  free entry in 2 a wkly comp to win fa cup fina...
3   ham  u dun say so early hor... u c already then say...
4   ham  nah i don't think he goes to usf, he lives aro...

New dataset shape: (5169, 2)


In [6]:
X = df["message"]
y = df["label"]

print("Input (X):")
print(X.head())

print("\nOutput (y):")
print(y.head())

Input (X):
0    go until jurong point, crazy.. available only ...
1                        ok lar... joking wif u oni...
2    free entry in 2 a wkly comp to win fa cup fina...
3    u dun say so early hor... u c already then say...
4    nah i don't think he goes to usf, he lives aro...
Name: message, dtype: object

Output (y):
0     ham
1     ham
2    spam
3     ham
4     ham
Name: label, dtype: object


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [8]:
tfidf = TfidfVectorizer(
    stop_words="english"
)

X_tfidf = tfidf.fit_transform(X)

print("Original number of messages:", len(X))
print("TF-IDF matrix shape:", X_tfidf.shape)

Original number of messages: 5169
TF-IDF matrix shape: (5169, 8444)


In [9]:
feature_names = tfidf.get_feature_names_out()

print("First 20 features:")
print(feature_names[:20])

First 20 features:
['00' '000' '000pes' '008704050406' '0089' '0121' '01223585236'
 '01223585334' '0125698789' '02' '0207' '02072069400' '02073162414'
 '02085076972' '021' '03' '04' '0430' '05' '050703']


In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (4135, 8444)
Testing data: (1034, 8444)


In [11]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()

model.fit(X_train, y_train)

print("Model trained successfully!")

Model trained successfully!


In [12]:
y_pred = model.predict(X_test)

print("Predictions completed!")
print(y_pred[:20])

Predictions completed!
['ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham'
 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham']


In [13]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)
print("Accuracy Percentage:", accuracy * 100, "%")

Accuracy: 0.9448742746615088
Accuracy Percentage: 94.48742746615088 %


In [14]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         ham       0.94      1.00      0.97       903
        spam       0.97      0.58      0.73       131

    accuracy                           0.94      1034
   macro avg       0.96      0.79      0.85      1034
weighted avg       0.95      0.94      0.94      1034



In [15]:
new_messages = [
    "Congratulations! You have won a free prize. Call now to claim it."
]

new_messages_tfidf = tfidf.transform(new_messages)

prediction = model.predict(new_messages_tfidf)

print("Message:", new_messages[0])
print("Predicted Category:", prediction[0])

Message: Congratulations! You have won a free prize. Call now to claim it.
Predicted Category: spam


In [16]:
new_messages = [
    "Hey, are you coming to class tomorrow?"
]

new_messages_tfidf = tfidf.transform(new_messages)

prediction = model.predict(new_messages_tfidf)

print("Message:", new_messages[0])
print("Predicted Category:", prediction[0])

Message: Hey, are you coming to class tomorrow?
Predicted Category: ham
